# 02 — Schaefer Atlas Surface Projection

This notebook:
- Loads the Schaefer 2018 parcellation (no local data needed — downloads automatically via nilearn)
- Projects the full atlas onto the inflated fsaverage surface
- Lets you interactively highlight individual ROIs or whole networks

**Dependencies:** `nilearn`, `nibabel`, `numpy`, `matplotlib`, `ipywidgets`

## 0 — Configuration

In [ ]:
N_ROIS           = 100   # 100, 200, 300, 400 ...
YEO_NETWORKS     = 7     # 7 or 17
ATLAS_RES_MM     = 2     # 1 or 2

## 1 — Imports & Helper Functions

In [ ]:
import numpy as np
import nibabel as nib
from nilearn import plotting, image, surface, datasets
import matplotlib.pyplot as plt
from ipywidgets import interact
from datetime import datetime

now = datetime.now()

In [ ]:
def roi_pattern_to_nifti(values, atlas_img):
    """Map a vector of per-ROI values back onto the atlas NIfTI volume.

    Args:
        values    : 1-D array of length N_ROIS
        atlas_img : the loaded atlas NIfTI image
    Returns:
        Nifti1Image with the same affine/header as atlas_img
    """
    atlas_data = atlas_img.get_fdata()
    new_data = np.zeros_like(atlas_data)
    for roi_label in np.unique(atlas_data):
        if roi_label == 0:
            continue
        idx = int(roi_label) - 1
        if idx < len(values):
            new_data[atlas_data == roi_label] = values[idx]
    return nib.Nifti1Image(new_data, affine=atlas_img.affine, header=atlas_img.header)


def nifti_to_surface(nifti_image):
    """Project a volumetric NIfTI onto fsaverage left/right surfaces."""
    fsaverage = datasets.fetch_surf_fsaverage()
    texture_left  = surface.vol_to_surf(nifti_image, fsaverage.pial_left,
                                        interpolation='linear', radius=3.0, n_samples=20)
    texture_right = surface.vol_to_surf(nifti_image, fsaverage.pial_right,
                                        interpolation='linear', radius=3.0, n_samples=20)
    for tex in (texture_left, texture_right):
        tex[(tex >= -0.0005) & (tex <= 0.0005)] = np.nan
    return texture_left, texture_right


def surface_plot(texture_left, texture_right, title='', colorbarlabel='',
                 min_val=-1, max_val=1, cmap='Spectral_r',
                 views=('lateral', 'medial'), save=False):
    """Four-panel inflated surface plot (lateral+medial, left+right)."""
    fsaverage = datasets.fetch_surf_fsaverage()
    fig, axes = plt.subplots(1, 4, figsize=(15, 3), subplot_kw={'projection': '3d'})
    fig.suptitle(title, fontsize=12, x=0.45)

    combos = [(views[0], 'left'), (views[0], 'right'),
              (views[1], 'left'), (views[1], 'right')]
    for ax, (view, hemi) in zip(axes, combos):
        texture = texture_left if hemi == 'left' else texture_right
        plotting.plot_surf_stat_map(
            fsaverage.infl_left if hemi == 'left' else fsaverage.infl_right,
            texture, hemi=hemi, view=view, colorbar=False,
            bg_map=fsaverage.sulc_left if hemi == 'left' else fsaverage.sulc_right,
            cmap=cmap, axes=ax, vmax=max_val, vmin=min_val, bg_on_data=True, darkness=None
        )

    fig.subplots_adjust(top=1, right=0.85, wspace=0, hspace=0, left=0.05, bottom=0)
    cbar_ax = fig.add_axes([0.95, 0.05, 0.02, 0.8])
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=min_val, vmax=max_val))
    sm.set_array([])
    plt.colorbar(sm, cax=cbar_ax).set_label(colorbarlabel, rotation=270, labelpad=15)

    if save:
        fname = str(now).replace(' ', '_').replace(':', '_').split('.')[0] + f'_{title}.png'
        plt.savefig(fname, transparent=True, dpi=300)
    plt.show()

## 2 — Load Atlas

In [ ]:
schaefer = datasets.fetch_atlas_schaefer_2018(
    n_rois=N_ROIS, yeo_networks=YEO_NETWORKS, resolution_mm=ATLAS_RES_MM
)
atlas_img    = image.load_img(schaefer['maps'])
labels       = schaefer['labels']

if isinstance(labels[0], bytes):
    labels = [l.decode('utf-8') for l in labels]

network_labels = np.unique([l.split('_')[2] for l in labels])

print(f'Atlas shape : {atlas_img.shape}')
print(f'ROIs        : {N_ROIS}')
print(f'Networks    : {list(network_labels)}')

## 3 — Full Atlas on Surface

In [ ]:
texture_left, texture_right = nifti_to_surface(atlas_img)
surface_plot(texture_left, texture_right,
             title='Schaefer Atlas', colorbarlabel='ROI index',
             min_val=0, max_val=N_ROIS, save=False)

## 4 — Highlight a Single ROI

Use the dropdown to pick a region, then run the next cell.

In [ ]:
chosen_roi = labels[0]   # default; widget below will update this

@interact(roi=labels)
def _pick_roi(roi):
    global chosen_roi
    chosen_roi = roi
    print(f'Selected: {roi}')

In [ ]:
roi_vec = np.zeros(N_ROIS)
roi_vec[labels.index(chosen_roi)] = 1
roi_nii = roi_pattern_to_nifti(roi_vec, atlas_img)
tl, tr  = nifti_to_surface(roi_nii)
surface_plot(tl, tr, title=chosen_roi, colorbarlabel='', min_val=0, max_val=0.5)

## 5 — Highlight a Whole Network

In [ ]:
chosen_network = network_labels[0]

@interact(network=list(network_labels))
def _pick_network(network):
    global chosen_network
    chosen_network = network
    print(f'Selected: {network}')

In [ ]:
net_vec = np.array([1.0 if chosen_network in l else 0.0 for l in labels])
net_nii = roi_pattern_to_nifti(net_vec, atlas_img)
tl, tr  = nifti_to_surface(net_nii)
surface_plot(tl, tr, title=chosen_network, colorbarlabel='', min_val=0, max_val=0.5)